# 03 - Select Top Democratic and Republican Candidates

This notebook selects one Democrat and one Republican candidate per state based on highest total receipts.
We also support manual overrides from `config/manual_overrides.yml`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))

from midterm_money.candidate_selection import load_manual_overrides, select_top_candidates
from midterm_money.normalize import normalize_party

In [ ]:
finance_path = ROOT / "data" / "processed" / "senate_candidate_finance_totals_2026.csv"
df_finance = pd.read_csv(finance_path, dtype=str)
print("Loaded finance totals shape:", df_finance.shape)
df_finance.head()

In [ ]:
overrides_path = ROOT / "config" / "manual_overrides.yml"
overrides = load_manual_overrides(overrides_path)
print("Manual overrides loaded:", overrides_path)
print(overrides)

In [ ]:
df_selected = select_top_candidates(df_finance, overrides=overrides)
df_selected = df_selected[
    ["state", "fec_candidate_id", "candidate_name", "committee_id", "total_receipts", "total_disbursements", "cash_on_hand_end_period", "cash_on_hand", "debts_owed_by_committee", "coverage_end_date", "party", "party_normalized", "selected_candidate", "selection_method", "selection_rank_within_state_party", "manual_override_reason"]
]
df_selected = df_selected.rename(columns={"candidate_name": "selected_candidate_name"})
print("Selected candidates shape:", df_selected.shape)
df_selected.head()


In [ ]:
selected_path = ROOT / "data" / "processed" / "senate_top_dem_rep_candidates_2026.csv"
outputs_path = ROOT / "outputs" / "senate_top_dem_rep_candidates_2026.csv"
selected_path.parent.mkdir(parents=True, exist_ok=True)
outputs_path.parent.mkdir(parents=True, exist_ok=True)
df_selected.to_csv(selected_path, index=False)
df_selected.to_csv(outputs_path, index=False)
print("Saved top candidate selection to:", outputs_path)

In [ ]:
wide = df_selected[df_selected["selected_candidate"]].copy()
wide = wide.pivot(index="state", columns="party_normalized", values=["selected_candidate_name", "fec_candidate_id", "committee_id", "total_receipts"])
wide.columns = [f"{col[0].lower()}_{col[1].lower()}" for col in wide.columns] if hasattr(wide.columns, "levels") else wide.columns
wide = wide.reset_index()
wide_path = ROOT / "outputs" / "senate_two_party_race_universe_2026.csv"
wide.to_csv(wide_path, index=False)
print("Saved two-party race universe to:", wide_path)
wide.head()